# Autoatención y posición

**Capítulo 5 · Universidad de las Hespérides**

Adaptación al español de *Dive into Deep Learning*, Aston Zhang, Zachary C. Lipton, Mu Li y Alexander J. Smola.
Fuente: `locked/chapter_attention-mechanisms-and-transformers/self-attention-and-positional-encoding.ipynb` · [Lección original](https://d2l.ai/chapter_attention-mechanisms-and-transformers/self-attention-and-positional-encoding.html).
Texto adaptado bajo [CC BY-SA 4.0](https://creativecommons.org/licenses/by-sa/4.0/). [Procedencia y cambios](../PROCEDENCIA.md).
Se conserva la secuencia de las celdas y de los ejercicios; las notas de Hespérides se identifican expresamente.

**Entorno:** ejecuta `uv sync` en la raíz y selecciona su Python como kernel. Las descargas se realizan una vez y quedan en `data/`.
Por defecto, el soporte limita los entrenamientos de `Trainer` a tres épocas y 1024/256 ejemplos para CPU.
Para repetir el régimen completo, inicia Jupyter con `HESPERIDES_COMPLETO=1`. Los ejemplos visuales pequeños conservan su propia configuración explícita.
Los datos de texto en inglés o francés son entradas de los experimentos originales y mantienen su idioma.


In [ ]:
from pathlib import Path
import sys
RAIZ = Path.cwd() if (Path.cwd() / "laboratorio").exists() else Path.cwd().parent
if str(RAIZ) not in sys.path:
    sys.path.insert(0, str(RAIZ))
from laboratorio import d2l, configurar, epocas
configurar()


# Autoatención y codificación posicional
<a id="sec_self-attention-and-positional-encoding"></a>

En el aprendizaje profundo, a menudo usamos CNNs o RNNs para codificar secuencias. Ahora, con los mecanismos de atención en mente, imagine alimentar una secuencia de tokens en un mecanismo de atención tal que en cada paso, cada token tiene su propia consulta, claves y valores. Aquí, al calcular el valor de la representación de un token en la siguiente capa, el token puede asistir (a través de su vector de consulta) a cualquier otro token (combinación basada en sus vectores clave). Usando el conjunto completo de puntuaciones de compatibilidad de la clave de consulta, podemos calcular, para cada token, una representación construyendo la suma ponderada apropiada sobre las otras fichas. Porque cada token está asistiendo a cada otra token (a diferencia del caso en que los pasos de decodificadores asisten a los pasos de encoder), tales arquitecturas se describen típicamente como *autoatención* modelos [Lin.Feng.Santos.ea.2017,Vaswani.Shazeer.Parmar.ea.2017](https://d2l.ai/chapter_references/zreferences.html), y en otra parte se describe como *intra-atención* modelo [Cheng.Dong.Lapata.2016,Parikh.Tackstrom.Das.ea.2016,Paulus.Xiong.Socher.2017](https://d2l.ai/chapter_references/zreferences.html). En esta sección, discutiremos la codificación de secuencias utilizando autoatención, incluyendo el uso de información adicional para el orden de secuencia.


In [ ]:
import math
import torch
from torch import nn
from laboratorio import d2l

## Autoatención

Dada una secuencia de tokens de entrada $\mathbf{x}_1, \ldots, \mathbf{x}_n$ donde cualquier $\mathbf{x}_i \in \mathbb{R}^d$ ($1 \leq i \leq n$), su autoatención produce una secuencia de la misma longitud $\mathbf{y}_1, \ldots, \mathbf{y}_n$, donde

$$\mathbf{y}_i = f(\mathbf{x}_i, (\mathbf{x}_1, \mathbf{x}_1), \ldots, (\mathbf{x}_n, \mathbf{x}_n)) \in \mathbb{R}^d$$

de acuerdo con la definición de agrupación de la atención en
[Referencia eq_attention_pooling](https://d2l.ai/#eq-attention-pooling).
Usando la atención multicabeza, el siguiente fragmento de código calcula la autoatención de un tensor con forma (tamaño del lote, número de pasos de tiempo o longitud de secuencia en tokens, $d$). El tensor de salida tiene la misma forma.


In [ ]:
num_hiddens, num_heads = 100, 5
attention = d2l.MultiHeadAttention(num_hiddens, num_heads, 0.5)
batch_size, num_queries, valid_lens = 2, 4, torch.tensor([3, 2])
X = torch.ones((batch_size, num_queries, num_hiddens))
d2l.check_shape(attention(X, X, X, valid_lens),
                (batch_size, num_queries, num_hiddens))

## Comparación de CNN, RNN y autoatención
<a id="subsec_cnn-rnn-self-attention"></a>

Comparemos arquitecturas para mapear una secuencia de tokens $n$ con otra de igual longitud, donde cada token de entrada o salida está representado por un vector $d$-dimensional. Específicamente, vamos a considerar CNNs, RNNs y autoatención. Compararemos su complejidad computacional, operaciones secuenciales y longitudes máximas de ruta. Tenga en cuenta que las operaciones secuenciales impiden el cálculo paralelo, mientras que una ruta más corta entre cualquier combinación de posiciones secuenciales hace más fácil aprender dependencias de largo alcance dentro de la secuencia [Hochreiter.Bengio.Frasconi.ea.2001](https://d2l.ai/chapter_references/zreferences.html).

![Comparación de CNN (sin mostrar padding), RNN y autoatención.](../recursos/originales/cnn-rnn-self-attention.svg)
<a id="fig_cnn-rnn-self-attention"></a>

Consideremos cualquier secuencia de texto como una "imagen unidimensional". Del mismo modo, las CNNs unidimensionales pueden procesar características locales como $n$-grams en texto. Dada una secuencia de longitud $n$, consideremos una capa convolucional cuyo tamaño del núcleo es $k$, y cuyo número de canales de entrada y salida son ambos $d$. La complejidad computacional de la capa convolucional es $\mathcal{O}(knd^2)$. Como muestra [Referencia fig_cnn-rnn-self-attention](https://d2l.ai/chapter_attention-mechanisms-and-transformers/self-attention-and-positional-encoding.html#fig-cnn-rnn-self-attention), las CNNs son jerárquicas, así que hay operaciones secuenciales $\mathcal{O}(1)$ y la longitud máxima de ruta es $\mathcal{O}(n/k)$. Por ejemplo, $\mathbf{x}_1$ y $\mathbf{x}_5$ están dentro del campo receptivo de una CNN de dos capas con tamaño de núcleo 3 en [Referencia fig_cnn-rnn-self-attention](https://d2l.ai/chapter_attention-mechanisms-and-transformers/self-attention-and-positional-encoding.html#fig-cnn-rnn-self-attention).

Al actualizar el estado oculto de las RNNs, la multiplicación de la matriz de peso $d \times d$ y el estado oculto $d$-dimensional tiene una complejidad computacional de $\mathcal{O}(d^2)$. Dado que la longitud de la secuencia es $n$, la complejidad computacional de la capa recurrente es $\mathcal{O}(nd^2)$. De acuerdo con [Referencia fig_cnn-rnn-self-attention](https://d2l.ai/chapter_attention-mechanisms-and-transformers/self-attention-and-positional-encoding.html#fig-cnn-rnn-self-attention), hay operaciones secuenciales $\mathcal{O}(n)$ que no pueden ser paralelizados y la longitud máxima de ruta también es $\mathcal{O}(n)$.

En autoatención, las consultas, claves y valores son todas matrices $n \times d$. Considere la atención por producto escalar escalado en
[Referencia eq_softmax_QK_V](https://d2l.ai/#eq-softmax-QK-V),
donde una matriz $n \times d$ se multiplica por una matriz $d \times n$, entonces la matriz de salida $n \times n$ se multiplica por una matriz $n \times d$. Como resultado, la autoatención tiene una complejidad computacional $\mathcal{O}(n^2d)$. Como podemos ver desde [Referencia fig_cnn-rnn-self-attention](https://d2l.ai/chapter_attention-mechanisms-and-transformers/self-attention-and-positional-encoding.html#fig-cnn-rnn-self-attention), cada token está directamente conectado a cualquier otro token vía autoatención. Por lo tanto, el cálculo puede ser paralelo con las operaciones secuenciales $\mathcal{O}(1)$ y la longitud máxima del camino también es $\mathcal{O}(1)$.

Con todo, tanto las CNN como la autoatención disfrutan de computación paralela y la autoatención tiene la longitud máxima de trayectoria más corta. Sin embargo, la complejidad computacional cuadrática con respecto a la longitud de secuencia hace la autoatención prohibitivamente lenta para secuencias muy largas.

## Codificación posicional
<a id="subsec_positional-encoding"></a>

A diferencia de las RNN, que procesan recurrentemente tokens de una secuencia uno por uno, la autoatención deja las operaciones secuenciales a favor de la computación paralela. Tenga en cuenta que la autoatención por sí misma no preserva el orden de la secuencia. ¿Qué hacemos si realmente importa que el modelo sepa en qué orden llegó la secuencia de entrada?

El enfoque dominante para preservar la información sobre el orden de los tokens es representar esto al modelo como una entrada adicional asociada a cada token. Estas entradas se llaman *codificación posicional*, y pueden ser aprendidas o fijas *a priori*. Ahora describimos un esquema simple para codificaciones posicionales fijas basadas en funciones seno y coseno [Vaswani.Shazeer.Parmar.ea.2017](https://d2l.ai/chapter_references/zreferences.html).

Supongamos que la representación de entrada $\mathbf{X} \in \mathbb{R}^{n \times d}$ contiene las embeddings dimensionales $d$ para tokens $n$ de una secuencia. La codificación posicional sale $\mathbf{X} + \mathbf{P}$ usando una matriz de incrustación posicional $\mathbf{P} \in \mathbb{R}^{n \times d}$ de la misma forma, cuyo elemento en la fila $i^\textrm{th}$ y la columna $(2j)^\textrm{th}$ o $(2j + 1)^\textrm{th}$ es

$$\begin{aligned} p_{i, 2j} &= \sin\left(\frac{i}{10000^{2j/d}}\right),\\p_{i, 2j+1} &= \cos\left(\frac{i}{10000^{2j/d}}\right).\end{aligned}$$

:eqlabel:`eq_positional-encoding-def`

A primera vista, este diseño de función trigonométrica se ve raro. Antes de dar explicaciones de este diseño, vamos a implementarlo primero en la siguiente clase `PositionalEncoding`.


In [ ]:
class PositionalEncoding(nn.Module):  #@save
    """Codificación posicional."""
    def __init__(self, num_hiddens, dropout, max_len=1000):
        super().__init__()
        self.dropout = nn.Dropout(dropout)
        # Crear un P lo suficientemente largo
        self.P = torch.zeros((1, max_len, num_hiddens))
        X = torch.arange(max_len, dtype=torch.float32).reshape(
            -1, 1) / torch.pow(10000, torch.arange(
            0, num_hiddens, 2, dtype=torch.float32) / num_hiddens)
        self.P[:, :, 0::2] = torch.sin(X)
        self.P[:, :, 1::2] = torch.cos(X)

    def forward(self, X):
        X = X + self.P[:, :X.shape[1], :].to(X.device)
        return self.dropout(X)

En la matriz de incrustación posicional $\mathbf{P}$, **filas corresponden a posiciones dentro de una secuencia y columnas representan diferentes dimensiones de codificación posicional**. En el ejemplo siguiente, podemos ver que las columnas $6^{\textrm{th}}$ y $7^{\textrm{th}}$ de la matriz de incrustación posicional tienen una frecuencia mayor que las columnas $8^{\textrm{th}}$ y $9^{\textrm{th}}$. El desplazamiento entre las columnas $6^{\textrm{th}}$ y $7^{\textrm{th}}$ (igual para la $8^{\textrm{th}}$ y la $9^{\textrm{th}}$) se debe a la alternancia de las funciones seno y coseno.


### Nota docente de Hespérides

Escribe qué información puede ver cada posición. Una máscara causal impide consultar el futuro; una máscara de padding excluye posiciones que no son datos. Comprueba que cada fila de atención suma uno antes de aplicar dropout. Los mapas de atención describen mezclas de valores, pero por sí solos no prueban una explicación causal del modelo.

Vínculo con los apuntes: sesión 5, «Autoatención y posición».


In [ ]:
encoding_dim, num_steps = 32, 60
pos_encoding = PositionalEncoding(encoding_dim, 0)
X = pos_encoding(torch.zeros((1, num_steps, encoding_dim)))
P = pos_encoding.P[:, :X.shape[1], :]
d2l.plot(torch.arange(num_steps), P[0, :, 6:10].T, xlabel='Fila (posición)',
         figsize=(6, 2.5), legend=["Col %d" % d for d in torch.arange(6, 10)])

### Información de posición absoluta
Para ver cómo la frecuencia monótonamente disminuida a lo largo de la dimensión de codificación se relaciona con la información posicional absoluta, imprimamos **las representaciones binarias** de $0, 1, \ldots, 7$. Como podemos ver, el bit más bajo, el segundo bit más bajo y el bit más bajo se alternan en cada número, cada dos números y cada cuatro números, respectivamente.


In [ ]:
for i in range(8):
    print(f'{i} in binary is {i:>03b}')

En las representaciones binarias, un bit más alto tiene una frecuencia menor que un bit más bajo. Del mismo modo, como se demuestra en el mapa de calor de abajo, **la codificación posicional disminuye frecuencias a lo largo de la dimensión de codificación** mediante el uso de funciones trigonométricas.


In [ ]:
P = P[0, :, :].unsqueeze(0).unsqueeze(0)
d2l.show_heatmaps(P, xlabel='Columna (dimensión)',
                  ylabel='Fila (posición)', figsize=(3.5, 4), cmap='Blues')

### Información relativa sobre la posición
Además de capturar información posicional absoluta, la codificación posicional anterior también permite a un modelo aprender fácilmente a asistir por posiciones relativas. Esto se debe a que para cualquier posición fija offset $\delta$, la codificación posicional en la posición $i + \delta$ puede ser representada por una proyección lineal de la posición $i$.

Esta proyección se puede explicar matemáticamente. Denotando $\omega_j = 1/10000^{2j/d}$, cualquier par de $(p_{i, 2j}, p_{i, 2j+1})$ en [Referencia eq_positional-encoding-def](https://d2l.ai/#eq-positional-encoding-def) se puede proyectar linealmente a $(p_{i+\delta, 2j}, p_{i+\delta, 2j+1})$ para cualquier desplazamiento fijo $\delta$:

$$\begin{aligned}
\begin{bmatrix} \cos(\delta \omega_j) & \sin(\delta \omega_j) \\  -\sin(\delta \omega_j) & \cos(\delta \omega_j) \\ \end{bmatrix}
\begin{bmatrix} p_{i, 2j} \\  p_{i, 2j+1} \\ \end{bmatrix}
=&\begin{bmatrix} \cos(\delta \omega_j) \sin(i \omega_j) + \sin(\delta \omega_j) \cos(i \omega_j) \\  -\sin(\delta \omega_j) \sin(i \omega_j) + \cos(\delta \omega_j) \cos(i \omega_j) \\ \end{bmatrix}\\
=&\begin{bmatrix} \sin\left((i+\delta) \omega_j\right) \\  \cos\left((i+\delta) \omega_j\right) \\ \end{bmatrix}\\
=& 
\begin{bmatrix} p_{i+\delta, 2j} \\  p_{i+\delta, 2j+1} \\ \end{bmatrix},
\end{aligned}$$

cuando la matriz de proyección $2\times 2$ no dependa de ningún índice de posición $i$.

## Resumen
En la autoatención, todas las consultas, claves y valores vienen del mismo lugar. Tanto CNNs como la autoatención disfrutan de computación paralela y la autoatención tiene la longitud máxima de ruta más corta. Sin embargo, la complejidad computacional cuadrática con respecto a la longitud de secuencia hace la autoatención prohibitivamente lenta para secuencias muy largas. Para usar la información de orden de secuencias, podemos inyectar información absoluta o relativa posicional agregando codificación posicional a las representaciones de entrada.

## Ejercicios
1. Supongamos que diseñamos una arquitectura profunda para representar una secuencia apilando capas de autoatención con codificación posicional. ¿Cuáles podrían ser los posibles problemas?
1. ¿Puede diseñar un método de codificación posicional aprendible?
1. ¿Podemos asignar diferentes embeddings aprendidas según diferentes compensaciones entre consultas y claves que se comparan en la autoatención? Consejo: puede referirse a las embeddings de posición relativa [shaw2018self,huang2018music](https://d2l.ai/chapter_references/zreferences.html).


[Debate del original](https://discuss.d2l.ai/t/1652)
